# Fine-tune intfloat/multilingual-e5-base với TripletLoss

Dataset: `train_v2.jsonl` (specific + vague queries, 1 hard negative/record)

## 1. Cài đặt thư viện

In [ ]:
# !pip install -q sentence-transformers datasets torch pandas numpy

## 2. Import & Config

In [ ]:
import json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
    evaluation,
)
from datasets import Dataset

from IPython.display import clear_output, display

# === CONFIG ===
BASE_MODEL     = "intfloat/multilingual-e5-base"
MAX_SEQ_LENGTH = 512

# Dataset — sử dụng bản v2 đã clean + vague supplement
DATA_DIR   = Path("embedding_project/data")
TRAIN_JSON = DATA_DIR / "train_v2.jsonl"
VALID_JSON = DATA_DIR / "valid_v2.jsonl"
TEST_JSON  = DATA_DIR / "test_v2.jsonl"
CORPUS_CSV = DATA_DIR / "Dataset_DATN_28k.csv"

# Training hyperparams
EPOCHS      = 2
BATCH_SIZE  = 8
LR          = 1e-5
WARMUP_RATIO= 0.1
TRIPLET_MARGIN = 0.2

OUTPUT_DIR  = Path("embedding_project/models/e5_base_v2_finetuned")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Base model: {BASE_MODEL}")
print(f"Train: {TRAIN_JSON}  |  Valid: {VALID_JSON}")

## 3. Load Dataset

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_raw = load_jsonl(TRAIN_JSON)
valid_raw = load_jsonl(VALID_JSON)
test_raw  = load_jsonl(TEST_JSON)

print(f"Train: {len(train_raw):,}  |  Valid: {len(valid_raw):,}  |  Test: {len(test_raw):,}")

# Thống kê nhanh
for name, recs in [("Train", train_raw), ("Valid", valid_raw), ("Test", test_raw)]:
    qt = {}
    for r in recs:
        qt[r.get("query_type", "unknown")] = qt.get(r.get("query_type","unknown"), 0) + 1
    n_hard0 = sum(1 for r in recs if r.get("n_hard", 0) == 0)
    print(f"  {name}: qtype={qt}  |  n_hard=0: {n_hard0:,} ({n_hard0/len(recs)*100:.1f}%)")

## 4. Build TripletLoss Dataset

Format: `{anchor, positive, negative}`

- `anchor`     = query text (cần prefix `query: `)
- `positive`   = product searchable_text (cần prefix `passage: `)
- `negative`   = neg[0] — 1 hard negative cố định/record

In [ ]:
def build_triplets(records: list[dict], min_neg: bool = True) -> list[dict]:
    """
    Chuyển dataset thành triplet format cho TripletLoss.
    
    Args:
        records: raw records từ train_v2.jsonl
        min_neg: nếu True, bỏ qua records không có hard negative
    """
    triplets = []
    skipped = 0
    
    for r in records:
        query   = r.get("query", "").strip()
        pos_lst = r.get("pos", [])
        neg_lst = r.get("neg", [])
        
        # Lấy positive đầu tiên
        if not pos_lst:
            skipped += 1
            continue
        pos_text = pos_lst[0].strip()
        if not pos_text:
            skipped += 1
            continue
        
        # Lấy hard negative đầu tiên (nếu có)
        if not neg_lst:
            if min_neg:
                skipped += 1
                continue
            neg_text = pos_text  # placeholder, sẽ không ảnh hưởng nhiều
        else:
            neg_text = neg_lst[0].strip()
        
        # E5 yêu cầu prefix
        anchor    = f"query: {query}"
        positive  = f"passage: {pos_text}"
        negative  = f"passage: {neg_text}"
        
        triplets.append({
            "anchor":   anchor,
            "positive": positive,
            "negative": negative,
            # Giữ lại metadata để debug/evaluate
            "query":       r.get("query", ""),
            "product_id":  r.get("product_id", ""),
            "query_type": r.get("query_type", "specific"),
            "n_hard":     r.get("n_hard", 0),
            "n_easy":     r.get("n_easy", 0),
        })
    
    return triplets, skipped

train_triplets, skipped_train = build_triplets(train_raw, min_neg=True)
valid_triplets, skipped_valid = build_triplets(valid_raw, min_neg=True)
test_triplets,  skipped_test  = build_triplets(test_raw,  min_neg=False)  # giữ lại test không có neg

print(f"Train triplets: {len(train_triplets):,}  (skipped {skipped_train:,} vì không có hard negative)")
print(f"Valid triplets: {len(valid_triplets):,}  (skipped {skipped_valid:,})")
print(f"Test triplets:  {len(test_triplets):,}   (kept all for evaluation)")

# Sample
t = train_triplets[0]
print(f"\nSample triplet:")
print(f"  anchor:   {t['anchor'][:80]}...")
print(f"  positive: {t['positive'][:80]}...")
print(f"  negative: {t['negative'][:80]}...")
print(f"  qtype:    {t['query_type']}  |  n_hard={t['n_hard']}  |  n_easy={t['n_easy']}")

## 5. Tạo Dataset cho sentence-transformers

In [ ]:
# Tách metadata ra — chỉ giữ anchor/positive/negative cho ST
def to_st_dataset(triplets: list[dict]) -> Dataset:
    return Dataset.from_list([
        {"anchor": t["anchor"], "positive": t["positive"], "negative": t["negative"]}
        for t in triplets
    ])

train_ds = to_st_dataset(train_triplets)
valid_ds = to_st_dataset(valid_triplets)

print(f"Train Dataset: {len(train_ds)}")
print(f"Valid Dataset: {len(valid_ds)}")
print(f"Train features: {train_ds.features}")

## 6. Load Model

In [ ]:
print(f"Loading {BASE_MODEL}...")
model = SentenceTransformer(BASE_MODEL, device=DEVICE)
model.max_seq_length = MAX_SEQ_LENGTH
print(f"Model loaded.  max_seq_length={model.max_seq_length}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")

## 7. Triplet Evaluator

Dùng valid set để chọn best checkpoint mỗi epoch.

In [ ]:
triplet_evaluator = evaluation.TripletEvaluator(
    anchors=   [t["anchor"]   for t in valid_triplets[:2000]],   # limit để eval nhanh
    positives=[t["positive"] for t in valid_triplets[:2000]],
    negatives=[t["negative"] for t in valid_triplets[:2000]],
    name="valid_triplet",
    show_progress_bar=True,
)
print(f"Triplet evaluator ready ({len(valid_triplets[:2000]):,} samples)")

## 8. Training Arguments

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

args = SentenceTransformerTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=1,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    fp16=torch.cuda.is_available(),
    bf16=False,
    max_grad_norm=1.0,
    
    # Eval & Save
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,           # giữ 3 checkpoints
    load_best_model_at_end=True,  # auto chọn best dựa trên eval metric
    
    # Logging
    logging_steps=100,
    log_on_each_node=False,
    
    # Misc
    run_name="e5_base_v2_finetuned",
    report_to=["none"],
    seed=42,
    data_seed=42,
)

print("Training args:")
for k, v in [("epochs", EPOCHS), ("batch_size", BATCH_SIZE), ("lr", LR),
             ("warmup_ratio", WARMUP_RATIO), ("fp16", args.fp16)]:
    print(f"  {k}: {v}")

## 9. Loss & Trainer

In [ ]:
train_loss = losses.TripletLoss(
    model=model,
    triplet_margin=TRIPLET_MARGIN,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    loss=train_loss,
    evaluator=triplet_evaluator,
)

print("Trainer ready. Starting training...")

## 10. Train!

In [ ]:
t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0
print(f"\nTraining done in {elapsed/60:.1f} min")
print(f"Total steps: {train_result.metrics.get('train_steps_per_second', 0) * elapsed:.0f}")

## 11. Save Model

In [ ]:
FINAL_DIR = OUTPUT_DIR / "final"
model.save(str(FINAL_DIR))
print(f"Model saved to: {FINAL_DIR}")

# Lưu config metadata
meta = {
    "base_model":     BASE_MODEL,
    "epochs":         EPOCHS,
    "batch_size":     BATCH_SIZE,
    "learning_rate":  LR,
    "warmup_ratio":   WARMUP_RATIO,
    "triplet_margin": TRIPLET_MARGIN,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_samples":  len(train_triplets),
    "valid_samples":  len(valid_triplets),
    "train_time_min": round(elapsed / 60, 1),
    "loss": "TripletLoss",
    "dataset": str(TRAIN_JSON.name),
}
with open(FINAL_DIR / "finetune_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print("Metadata saved.")

## 12. Evaluate trên Test Set

Đánh giá retrieval quality bằng Recall@K, MRR@K, NDCG@K.

In [ ]:
# Load corpus
corpus_df = pd.read_csv(CORPUS_CSV, usecols=["product_id", "searchable_text"])
corpus_df = corpus_df.dropna(subset=["product_id", "searchable_text"])
corpus_df["product_id"] = corpus_df["product_id"].astype(str)
corpus_df = corpus_df.drop_duplicates(subset="product_id", keep="first")
corpus_ids   = corpus_df["product_id"].tolist()
corpus_texts = corpus_df["searchable_text"].tolist()

# Build pid→idx mapping
pid2idx = {pid: i for i, pid in enumerate(corpus_ids)}

print(f"Corpus: {len(corpus_ids):,} products")

In [ ]:
def normalize(v: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(v, axis=1, keepdims=True) + 1e-12
    return v / norm

def batch_encode(texts: list[str], batch_size: int = 256) -> np.ndarray:
    """Encode texts in batches."""
    emb = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return emb

# Encode corpus
print("Encoding corpus...")
t0 = time.time()
corpus_emb = batch_encode([f"passage: {t}" for t in corpus_texts])
print(f"Corpus encoded: {corpus_emb.shape}  ({time.time()-t0:.1f}s)")

In [ ]:
def recall_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:
    hits = sum(1 for pid in retrieved[:k] if pid in relevant)
    return hits / max(len(relevant), 1)

def mrr_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:
    for i, pid in enumerate(retrieved[:k]):
        if pid in relevant:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:
    dcg = 0.0
    for i, pid in enumerate(retrieved[:k]):
        if pid in relevant:
            dcg += 1.0 / np.log2(i + 2)
    # IDCG
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(k, len(relevant))))
    return dcg / max(idcg, 1e-9)

def retrieve_topk(query_texts: list[str], k: int = 50) -> list[list[str]]:
    """Retrieve top-k product IDs for queries."""
    q_emb = model.encode(
        [f"query: {q}" for q in query_texts],
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    scores = np.dot(q_emb, corpus_emb.T)  # cosine (đã normalize)
    topk_idx = np.argpartition(-scores, th=k, axis=1)[:, :k]
    # Sort by score descending
    sorted_idx = np.argsort(-np.take_along_axis(scores, topk_idx, axis=1), axis=1)
    topk_idx = np.take_along_axis(topk_idx, sorted_idx, axis=1)
    return [[corpus_ids[j] for j in row] for row in topk_idx]

def evaluate_split(triplets: list[dict], k_values: list[int]) -> dict:
    """
    Evaluate triplet list.
    Ground truth: mỗi query có đúng 1 relevant product = product_id trong triplet.
    """
    queries  = [t["query"] for t in triplets]
    labels   = [{t["product_id"]} for t in triplets]
    
    retrieved = retrieve_topk(queries, k=max(k_values))
    
    results = {}
    for k in k_values:
        rec_k  = [recall_at_k(r, lbl, k) for r, lbl in zip(retrieved, labels)]
        mrr_k  = [mrr_at_k(r, lbl, k)    for r, lbl in zip(retrieved, labels)]
        ndcg_k = [ndcg_at_k(r, lbl, k)   for r, lbl in zip(retrieved, labels)]
        results[f"Recall@{k}"]  = round(np.mean(rec_k)  * 100, 2)
        results[f"MRR@{k}"]    = round(np.mean(mrr_k)  * 100, 2)
        results[f"NDCG@{k}"]   = round(np.mean(ndcg_k) * 100, 2)
    
    return results

# Đánh giá test set
print(f"Evaluating on Test set ({len(test_triplets):,} samples)...")
t0 = time.time()
test_metrics = evaluate_split(test_triplets, k_values=[1, 5, 10, 20, 50])
print(f"Done in {time.time()-t0:.1f}s\n")

# In kết quả
print("=" * 45)
print("  TEST SET RETRIEVAL RESULTS")
print("=" * 45)
for metric, value in test_metrics.items():
    print(f"  {metric:20s}: {value:6.2f}%")
print("=" * 45)

In [ ]:
# Tách kết quả theo query_type
print("\nPer-query-type breakdown:")
print("-" * 55)
for qtype in ["specific", "vague"]:
    subset = [t for t in test_triplets if t["query_type"] == qtype]
    if not subset:
        continue
    print(f"\n  [{qtype.upper()}]  n={len(subset):,}")
    metrics = evaluate_split(subset, k_values=[10, 20])
    for metric, value in metrics.items():
        print(f"    {metric:20s}: {value:6.2f}%")

In [ ]:
# Lưu kết quả
RESULTS_FILE = FINAL_DIR / "test_metrics.json"
with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2, ensure_ascii=False)
print(f"Metrics saved to: {RESULTS_FILE}")

---

## Tổng kết

Sau khi train xong, model nằm ở:
```
embedding_project/models/e5_base_v2_finetuned/final/
```

Sử dụng model:
```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("embedding_project/models/e5_base_v2_finetuned/final")

# Query
query_emb  = model.encode(["query: sữa cho bé"], normalize_embeddings=True)
# Corpus
corpus_emb = model.encode([f"passage: {text}" for text in corpus_texts], normalize_embeddings=True)

# Retrieval
scores = np.dot(query_emb, corpus_emb.T)  # cosine similarity
top_k_idx = np.argsort(-scores[0])[:10]
```